In [1]:
import duckdb
import pandas as pd

# import os
# print(os.getcwd())

con = duckdb.connect()

structure de la colonne nutriment

In [2]:
con.sql("SELECT typeof(nutriments) FROM '../../data/food.parquet' LIMIT 1").show()

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                              typeof(nutriments)                                                                              │
│                                                                                   varchar                                                                                    │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ STRUCT("name" VARCHAR, "value" FLOAT, "100g" FLOAT, serving FLOAT, unit VARCHAR, prepared_value FLOAT, prepared_100g FLOAT, prepared_serving FLOAT, prepared_unit VARCHAR)[] │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Créer la vue nutriments_long, pour chaque produit (t.code), elle déplie la liste de nutriments (UNNEST) en une ligne par nutriment, et garde deux colonnes : le nom du nutriment (n.name) et sa valeur pour 100g (n."100g", renommée value_100g).

In [3]:
con.sql("""
    CREATE VIEW nutriments_long AS
    SELECT
        t.code,
        n.name,
        n."100g" AS value_100g
    FROM '../../data/food.parquet' AS t,
    UNNEST(t.nutriments) AS t2(n)
""")

Totalité des différents nutriments

In [4]:
df_names = con.sql("SELECT name, COUNT(*) as n FROM nutriments_long GROUP BY name ORDER BY 2 DESC").df()
print(df_names.to_string())

                                                                                          name        n
0                                                                                       energy  3374966
1                                                                                  energy-kcal  3366826
2                                                                                     proteins  3340804
3                                                                                carbohydrates  3338628
4                                                                                          fat  3335974
5                                                                                       sugars  3147562
6                                                                                saturated-fat  3085918
7                                                                                       sodium  2917759
8                                                               

Top 50 en nombres de lignes

In [5]:
con.sql("""
    SELECT name, COUNT(*) as n
    FROM nutriments_long
    GROUP BY name
    ORDER BY n DESC
    LIMIT 50
""").show(max_rows=50)

┌─────────────────────────────────────────────────────┬─────────┐
│                        name                         │    n    │
│                       varchar                       │  int64  │
├─────────────────────────────────────────────────────┼─────────┤
│ energy                                              │ 3374966 │
│ energy-kcal                                         │ 3366826 │
│ proteins                                            │ 3340804 │
│ carbohydrates                                       │ 3338628 │
│ fat                                                 │ 3335974 │
│ sugars                                              │ 3147562 │
│ saturated-fat                                       │ 3085918 │
│ sodium                                              │ 2917759 │
│ salt                                                │ 2917758 │
│ energy-kj                                           │ 2389174 │
│ fiber                                               │ 1787069 │
│ nova-gro

liste des nutriments clés

In [6]:
nutriments_cles = [
    'energy-kcal', 'fat', 'saturated-fat', 'carbohydrates',
    'sugars', 'fiber', 'proteins', 'salt', 'sodium'
]

Créer une vrai table en y mettant le résultat du SELECT, jointure implicite entre t ( chaque ligne de code, un produit ) et UNNEST(t.nutriments) la liste des nutriments de ce produit,filtré par les nutriment clés et regrouper par produit (code); MAX(CASE WHEN n.name = '...' THEN n."100g" END) AS ..._100g est un pivot manuel ( conditionnal aggregation ) pour pouvoir transformer les nutriment clé en colonne

### Attention temps d'éxécution long ( ~ 4min 30 )

In [8]:
con.sql("""
    CREATE TABLE nutriments_wide AS
    SELECT
        t.code,
        MAX(CASE WHEN n.name = 'energy-kcal' THEN n."100g" END) AS energy_kcal_100g,
        MAX(CASE WHEN n.name = 'fat' THEN n."100g" END) AS fat_100g,
        MAX(CASE WHEN n.name = 'saturated-fat' THEN n."100g" END) AS saturated_fat_100g,
        MAX(CASE WHEN n.name = 'carbohydrates' THEN n."100g" END) AS carbohydrates_100g,
        MAX(CASE WHEN n.name = 'sugars' THEN n."100g" END) AS sugars_100g,
        MAX(CASE WHEN n.name = 'fiber' THEN n."100g" END) AS fiber_100g,
        MAX(CASE WHEN n.name = 'proteins' THEN n."100g" END) AS proteins_100g,
        MAX(CASE WHEN n.name = 'salt' THEN n."100g" END) AS salt_100g,
        MAX(CASE WHEN n.name = 'sodium' THEN n."100g" END) AS sodium_100g
    FROM '../../data/food.parquet' AS t,
    UNNEST(t.nutriments) AS t2(n)
    WHERE n.name IN ('energy-kcal','fat','saturated-fat','carbohydrates','sugars','fiber','proteins','salt','sodium')
    GROUP BY t.code
""")

CatalogException: Catalog Error: Table with name "nutriments_wide" already exists!